# 1. Data Acquisition and Validation

**Module:** AI and Data Science (OIM7507-B)  
**Deliverable:** Data Acquisition Strategy (10 marks)

## 1.1 Objectives
- Describe the problem context and data requirements for UK housing price analysis.
- Justify the chosen data collection method (browser extension + backend).
- Document sources, tools, and processes with metadata for the raw dataset.

## 1.2 Problem Context & Data Needs

**Problem:** We need to analyse factors influencing UK residential property prices and prepare a dataset suitable for price prediction and market analysis (e.g. regression or classification models).

**Required variables:**
| Variable | Purpose |
|----------|---------|
| `price` | Target for prediction; key outcome variable |
| `area_sqft` | Strong predictor of value; normalisation (price per sqft) |
| `bedrooms`, `bathrooms`, `living_rooms` | Capacity and lifestyle indicators |
| `epc_rating` | Energy efficiency (A–G); regulatory and value relevance |
| `city` / postcode | Geographic variation and bias analysis |
| `address` | Location detail; postcode extraction for area-level stats |
| `description` | Text features (garden, parking, balcony, etc.) |
| `property_type` | Flat, Terraced, Detached, Semi-Detached, etc. |

**Quality requirements:**
- **Completeness:** Maximise non-null rates for price, bedrooms, city, property_type; document missingness for area_sqft and epc_rating.
- **Geographic coverage:** Multiple cities/regions to support generalisation and bias assessment.
- **Dataset size:** Target ≥12,000 records for robust ML and validation.
- **Consistency:** Numeric types for price and area; standardised EPC scale (A–G).

## 1.3 Justification of the Data Collection Pathway

Data was collected using a **browser extension** (`zoopla_extension`) with backend support (`zoopla_crawler_app`).

**Technical reasons for using an extension:**
1. **Anti-bot avoidance:** Zoopla uses anti-scraping measures; a real browser session with human-like behaviour reduces blocks.
2. **JavaScript-rendered content:** Listing details are often loaded dynamically; the extension runs in the page context and captures the final DOM.
3. **Cookies and session:** Uses the same session as a logged-in or normal user, avoiding CAPTCHAs and access restrictions.
4. **Realistic requests:** Traffic originates from a real browser, matching expected patterns.

**Limitations:**
- **Slower collection:** Manual or semi-automated browsing is slower than headless bulk requests.
- **Sampling bias:** Coverage depends on which pages were visited (e.g. London-heavy if London was browsed more).
- **Lower automation:** Requires human interaction or controlled automation.
- **Reproducibility:** Exact snapshot depends on date and browsing path; re-runs may differ.

**Mitigation strategies:**
- Rotate browsing sessions and document capture dates.
- Log metadata (user agent, timestamps) for reproducibility.
- Randomise navigation where possible (e.g. city order, pagination).
- Document sampling criteria (e.g. “listings visible on Zoopla for-sale in selected cities between dates X–Y”).

## 1.4 Sources, Tools, and Processes

**Technology stack:**
- **Python** (backend scripts, data handling)
- **pandas** (CSV aggregation, basic validation)
- **Browser extension** (JavaScript): captures listing fields from Zoopla pages and sends to backend
- **Backend aggregator** (e.g. Flask/FastAPI or script): receives payloads from extension, appends to CSV

**Process flowchart (conceptual):**

```
Zoopla listing pages (for-sale)
         ↓
Browser extension captures (price, address, beds, baths, area, EPC, description, etc.)
         ↓
Backend aggregator receives and validates
         ↓
Append to data/raw/zoopla_raw.csv
         ↓
Raw CSV dataset (used in downstream notebooks)
```

*To extend or recollect:* Run the extension on target cities/regions, ensure backend is logging to the same CSV (or merge CSVs with consistent columns), and record capture date and any filters (e.g. “for-sale only”).

## 1.5 Raw Dataset Metadata and Validation

Load the raw CSV and report:
- Number of rows and columns
- Column names and dtypes
- Capture date (from file timestamp or `created_at` if present)
- Brief sample of the data

In [1]:
import pandas as pd
import os
from pathlib import Path

# Path to raw data: run from project root (final_v1) or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_RAW = ROOT / "data" / "raw" / "zoopla_raw.csv"
df = pd.read_csv(DATA_RAW)

print("Raw dataset path:", DATA_RAW.resolve())
print("File exists:", DATA_RAW.exists())
if DATA_RAW.exists():
    stat = os.stat(DATA_RAW)
    import datetime
    print("File modified (capture date proxy):", datetime.datetime.fromtimestamp(stat.st_mtime))

Raw dataset path: /Users/hoamai/Personal/bradford/AI_AND_DATASCIENCE/final_v1/data/raw/zoopla_raw.csv
File exists: True
File modified (capture date proxy): 2026-03-08 08:14:49.033247


In [2]:
# Metadata: dimensions and columns
n_rows, n_cols = df.shape
print("Number of rows:", n_rows)
print("Number of columns:", n_cols)
print("\nColumn names and dtypes:")
print(df.dtypes)
print("\nNon-null counts:")
print(df.count())

Number of rows: 14438
Number of columns: 13

Column names and dtypes:
id               int64
url                str
city               str
price            int64
address            str
property_type      str
bedrooms         int64
bathrooms        int64
living_rooms     int64
area_sqft        int64
description        str
created_at         str
epc_rating         str
dtype: object

Non-null counts:
id               14438
url              14438
city             14438
price            14438
address          14438
property_type    14438
bedrooms         14438
bathrooms        14438
living_rooms     14438
area_sqft        14438
description      14438
created_at       14438
epc_rating       11406
dtype: int64


In [3]:
# Capture date from data if available (created_at)
if "created_at" in df.columns:
    sample_dates = pd.to_datetime(df["created_at"], errors="coerce").dropna()
    if len(sample_dates) > 0:
        print("Earliest created_at:", sample_dates.min())
        print("Latest created_at:", sample_dates.max())
print("\nFirst 3 rows (key columns):")
cols_show = [c for c in ["id", "city", "price", "property_type", "bedrooms", "area_sqft", "epc_rating"] if c in df.columns]
print(df[cols_show].head(3).to_string())

Earliest created_at: 2026-03-06 23:14:43.336000
Latest created_at: 2026-03-07 12:05:49.380000

First 3 rows (key columns):
   id    city   price   property_type  bedrooms  area_sqft epc_rating
0   1  London  600000  Flat/Apartment         3          0          B
1   2  London  375000  Flat/Apartment         2          0          B
2   3  London  695000  Flat/Apartment         2        645          C
